# Smart Shelf CV — เทียบรูป Reference (ถูกต้อง) กับรูปจริง

แนวทาง: ไม่ต้องพิมพ์ planogram เป็น JSON เอง — ถ่ายรูปชั้นวางตอนวางถูกต้อง 1 รูป (**reference**) แล้วเอาโมเดล YOLOv8 ที่ fine-tune เอง (`best.pt`) detect ทั้ง reference กับรูปจริงที่จะเช็ค (**test**) จากนั้นเทียบ **shape class** ที่ detect ได้ (`Bottle`/`Box`/`Carton`/`Jar`/`Pouch`/`Tube`/`can`) ทีละ grid cell

ก่อนรัน ต้องมี:
- ไฟล์ `best.pt` (โมเดลที่เทรนเสร็จจาก `smart_shelf_cv_train_yolov8.ipynb`) วางไว้ในโฟลเดอร์เดียวกับ notebook นี้
- รูป reference (ชั้นวางตอนถูกต้อง) 1 รูป
- รูป test (ชั้นวางที่จะเช็ค — เช่นเคสที่จงใจเอาของออก/สลับที่) 1 รูป
- **สำคัญ:** ถ่าย 2 รูปนี้จากมุมกล้อง/ระยะเดียวกันให้มากที่สุด ไม่งั้นตำแหน่ง grid จะเทียบกันไม่ตรง

**หมายเหตุ:** ไม่ต้องใช้ API key หรือเน็ตอีกต่อไป เพราะรันโมเดลในเครื่อง/Colab ตรงๆ (เร็วกว่าและไม่เสี่ยงเรื่อง network ตอน demo ตามที่คุยกันไว้)

## Step 1: ติดตั้ง library

In [1]:
!pip install ultralytics pillow matplotlib -q

## Step 2: ตั้งค่า

แก้ค่าตรงนี้ให้ตรงกับของจริงก่อนรัน

In [2]:
MODEL_PATH = 'Final_model\\best (2).pt'
CONFIDENCE_THRESHOLD = 0.4

REFERENCE_IMAGE_PATH = 'shelf_reference.jpg'
TEST_IMAGE_PATH = 'shelf_test.jpg'

GRID_ROWS = 2
GRID_COLS = 4

## Step 3: โหลดโมเดล + ฟังก์ชันหลัก (detect + แปลงเป็น grid)

โหลดโมเดลครั้งเดียว แล้วเขียนฟังก์ชัน `analyze_image` เรียกซ้ำได้ทั้งกับรูป reference และรูป test

In [3]:
from ultralytics import YOLO
from PIL import Image

model = YOLO(MODEL_PATH)

def analyze_image(image_path):
    result = model.predict(source=image_path, conf=CONFIDENCE_THRESHOLD, verbose=False)[0]
    image = Image.open(image_path).convert('RGB')
    img_w, img_h = image.size

    items = []
    for box in result.boxes:
        cx, cy, w, h = box.xywh[0].tolist()
        class_name = result.names[int(box.cls[0])]
        row = min(int(cy / img_h * GRID_ROWS), GRID_ROWS - 1)
        col = min(int(cx / img_w * GRID_COLS), GRID_COLS - 1)

        items.append({
            'center_x': cx, 'center_y': cy, 'width': w, 'height': h,
            'row': row, 'col': col,
            'class': class_name,
            'confidence': float(box.conf[0]),
        })
    return items, image, img_w, img_h

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\User\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## Step 4: รัน detection กับรูป reference และรูป test

In [4]:
reference_items, reference_image, ref_w, ref_h = analyze_image(REFERENCE_IMAGE_PATH)
print(f'Reference: เจอ {len(reference_items)} ชิ้น')

test_items, test_image, test_w, test_h = analyze_image(TEST_IMAGE_PATH)
print(f'Test: เจอ {len(test_items)} ชิ้น')

FileNotFoundError: shelf_reference.jpg does not exist

## Step 5: เทียบผล reference กับ test ทีละ grid cell

- **correct** — class ตรงกันทั้งสองรูป
- **misplaced** — cell นี้มีของทั้งคู่ แต่ class ไม่ตรงกัน (วางสินค้าผิดชนิด)
- **missing** — reference มีของ แต่ test cell นี้ว่าง (ของหาย/ยังไม่ได้เติม)
- **extra** — test มีของ แต่ reference cell นี้ไม่มี (วางของแปลกปลอมเพิ่ม)

In [ ]:
def to_grid_map(items):
    grid = {}
    for it in items:
        grid[(it['row'], it['col'])] = it
    return grid

reference_grid = to_grid_map(reference_items)
test_grid = to_grid_map(test_items)

all_cells = set(reference_grid.keys()) | set(test_grid.keys())

report = []
for row, col in sorted(all_cells):
    ref_item = reference_grid.get((row, col))
    test_item = test_grid.get((row, col))

    if ref_item and test_item:
        status = 'correct' if ref_item['class'] == test_item['class'] else 'misplaced'
    elif ref_item and not test_item:
        status = 'missing'
    else:
        status = 'extra'

    report.append({
        'row': row, 'col': col,
        'reference_class': ref_item['class'] if ref_item else None,
        'test_class': test_item['class'] if test_item else None,
        'status': status,
    })

for item in report:
    print(item)

## Step 6: วาดผลลัพธ์ทับรูป test (เขียว = ถูก, แดง = ผิด/ขาด/แปลกปลอม)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(1, figsize=(12, 8))
ax.imshow(test_image)

for det in test_items:
    left = det['center_x'] - det['width'] / 2
    top = det['center_y'] - det['height'] / 2
    rect = patches.Rectangle((left, top), det['width'], det['height'],
                              linewidth=2, edgecolor='cyan', facecolor='none')
    ax.add_patch(rect)
    ax.text(left, top - 5, det['class'], color='cyan', fontsize=9)

cell_w, cell_h = test_w / GRID_COLS, test_h / GRID_ROWS
for item in report:
    color = 'lime' if item['status'] == 'correct' else 'red'
    x, y = item['col'] * cell_w, item['row'] * cell_h
    rect = patches.Rectangle((x, y), cell_w, cell_h,
                              linewidth=2, edgecolor=color, facecolor='none', linestyle='--')
    ax.add_patch(rect)
    ax.text(x + 5, y + 20, item['status'], color=color, fontsize=10, weight='bold')

plt.axis('off')
plt.show()